# Phase 4 & 5 — RIFE Baseline + Frequency-Aware Fine-Tuning
**Runtime must be set to T4 GPU before running.**

In [ ]:
# CELL 1: Setup — clone repo + install deps
import os, sys
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SatelliteInterpolation')
REPO_DIR   = Path('/content/satellite-interpolation')
RIFE_DIR   = REPO_DIR / 'models' / 'rife'

# Clone project repo (skip if already done)
if not REPO_DIR.exists():
    !git clone https://github.com/YOUR_USERNAME/satellite-interpolation-frequency-aware.git /content/satellite-interpolation
else:
    print('Repo already cloned.')

# Clone Practical-RIFE
if not (RIFE_DIR / 'model').exists():
    !git clone --depth 1 https://github.com/hzwer/Practical-RIFE.git {RIFE_DIR}
else:
    print('RIFE already cloned.')

# Install dependencies
!pip install scikit-image imageio imageio-ffmpeg gdown -q
print('Setup done!')

In [ ]:
# CELL 2: Download RIFE pretrained weights
import gdown, shutil
from pathlib import Path

RIFE_DIR    = Path('/content/satellite-interpolation/models/rife')
WEIGHTS_DIR = RIFE_DIR / 'train_log'

if not WEIGHTS_DIR.exists() or not list(WEIGHTS_DIR.glob('*.pkl')):
    WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = RIFE_DIR / 'train_log.zip'
    gdown.download('https://drive.google.com/uc?id=1mUK9iON6Es14oK46-cTflJUex5nE_6Rl',
                   str(zip_path), quiet=False)
    shutil.unpack_archive(str(zip_path), str(RIFE_DIR))
    zip_path.unlink(missing_ok=True)
    print('Weights downloaded and extracted!')
    print(list(WEIGHTS_DIR.iterdir()))
else:
    print('Weights already present:', list(WEIGHTS_DIR.glob('*.pkl')))

In [ ]:
# CELL 3: Load data from Drive
import shutil
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SatelliteInterpolation')
LOCAL_DATA = Path('/content/satellite-interpolation/data')

# Copy train/val/test from Drive to local (faster I/O during training)
for split in ['train', 'validation', 'test']:
    src = DRIVE_ROOT / 'data' / split
    dst = LOCAL_DATA / split
    if src.exists() and not dst.exists():
        shutil.copytree(str(src), str(dst))
        print(f'Copied {split}: {len(list(dst.iterdir()))} triplets')
    elif dst.exists():
        print(f'{split}: already local ({len(list(dst.iterdir()))} triplets)')
    else:
        print(f'[WARN] {src} not found in Drive. Run preprocess.py first.')

In [ ]:
# CELL 4: Run RIFE Baseline (Phase 4)
import sys
sys.path.insert(0, '/content/satellite-interpolation/scripts')
os.chdir('/content/satellite-interpolation')

!python scripts/run_baseline.py --skip-setup --limit 30
# --limit 30 = evaluate first 30 test triplets (faster for demo)
# Remove --limit to evaluate all 120

In [ ]:
# CELL 5: Show baseline metrics
import json
from pathlib import Path

m_path = Path('/content/satellite-interpolation/outputs/metrics/baseline_metrics.json')
if m_path.exists():
    m = json.loads(m_path.read_text())
    print('=== BASELINE METRICS ===')
    print(f"Samples  : {m['n_samples']}")
    print(f"Avg PSNR : {m['avg_psnr']:.3f} dB")
    print(f"Avg SSIM : {m['avg_ssim']:.4f}")
    print(f"Avg MSE  : {m['avg_mse']:.2f}")

In [ ]:
# CELL 6: GPU Profile — measure memory + iter time (Phase 5 prep)
os.chdir('/content/satellite-interpolation')
!python scripts/train.py --profile-only --batch 4 --tile 256

In [ ]:
# CELL 7: Full Fine-Tuning (Phase 5)
# Adjust --batch and --tile based on GPU profile above
os.chdir('/content/satellite-interpolation')
!python scripts/train.py \
    --epochs 30 \
    --batch 4 \
    --tile 256 \
    --lr 1e-4 \
    --alpha 0.7 \
    --beta 0.3

In [ ]:
# CELL 8: Copy outputs back to Drive
import shutil
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SatelliteInterpolation')
LOCAL_ROOT = Path('/content/satellite-interpolation')

pairs = [
    (LOCAL_ROOT / 'models' / 'checkpoints',   DRIVE_ROOT / 'checkpoints'),
    (LOCAL_ROOT / 'models' / 'finetuned',     DRIVE_ROOT / 'models'),
    (LOCAL_ROOT / 'outputs' / 'baseline',     DRIVE_ROOT / 'outputs' / 'baseline'),
    (LOCAL_ROOT / 'outputs' / 'metrics',      DRIVE_ROOT / 'outputs' / 'metrics'),
    (LOCAL_ROOT / 'outputs' / 'animations',   DRIVE_ROOT / 'outputs' / 'animations'),
]

for src, dst in pairs:
    if src.exists():
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.rglob('*'):
            if f.is_file():
                rel = f.relative_to(src)
                (dst / rel).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(str(f), str(dst / rel))
        print(f'Synced: {src.name} -> Drive')

print('All outputs saved to Google Drive!')